# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, following best practices for referencing dataset entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we retrieve and display all available record sets and their fields by `@id`.

In [ ]:
# Inspect available record sets (by @id)
from pprint import pprint

record_set_ids = []

print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set.id}, name: {record_set.name}")
    record_set_ids.append(record_set.id)

print("\nFields for each RecordSet (by @id):")
for record_set in dataset.record_sets:
    print(f"\nRecordSet: {record_set.name} (@id: {record_set.id})")
    for field in record_set.fields:
        print(f"  - Field @id: {field.id}, name: {field.name}, type: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Each entity is referenced by its `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns:", df.columns.tolist())
    else:
        print("No records found.")

# If at least one DataFrame loaded, display summary
if dataframes:
    # Use the first loaded record set for exploration
    primary_rs_id = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows from RecordSet {primary_rs_id}:")
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filter records, normalize numeric fields, and group/categorize data by key attributes.

**Note:** All fields and columns are referenced by their `@id`.

In [ ]:
# Identify numeric fields from fields list of primary_rs_id
numeric_field_id = None
group_field_id = None
for record_set in dataset.record_sets:
    if record_set.id == primary_rs_id:
        for field in record_set.fields:
            if field.data_type in ('Float', 'Integer') and numeric_field_id is None:
                numeric_field_id = field.id
            if field.data_type == 'Text' and group_field_id is None:
                group_field_id = field.id

# Show which fields chosen
print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

df = dataframes[primary_rs_id]

# If a numeric field exists and is in DataFrame, proceed
if numeric_field_id and numeric_field_id in df.columns:
    try:
        # Convert to numeric (if string/object)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    except Exception as e:
        print(f"Error converting field {numeric_field_id}: {e}")
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].nunique() > 2 else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized column {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if it exists
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example histogram of the chosen numeric field, grouped by a category if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8,5))
    if group_field_id and group_field_id in df.columns:
        sns.histplot(data=df, x=numeric_field_id, hue=group_field_id, element='step', bins=15)
        plt.title(f"Distribution of {numeric_field_id} grouped by {group_field_id}")
    else:
        sns.histplot(df[numeric_field_id].dropna(), bins=15)
        plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we demonstrated the complete workflow for loading a Croissant-specified dataset using `mlcroissant`, exploring its metadata, extracting records by `@id`, performing basic exploratory analysis, and visualizing core distributions. All references to record sets and fields followed the `@id` conventions for consistency and reproducibility. You can extend this workflow to more complex EDA and modeling tasks as needed.